In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

In [ ]:
# Cell 1: Setup
!pip install -q bert_score transformers sentence-transformers

import json
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import torch

print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Load Dataset
# Change path based on which dataset you're testing
DATASET = "hotpotqa"  # or "squad"

if DATASET == "hotpotqa":
    with open(f'{WORK_DIR}/gemma/hotpot/hotpot_qwen25_rag_length_controlled_1000_binmatched.json') as f:
        dataset = json.load(f)
else:
    with open(f'{WORK_DIR}/gemma/squad/hotpot_qwen25_rag_length_controlled_1000_binmatched.json') as f:
        dataset = json.load(f)

labels = np.array([s['hallucinated'] for s in dataset])
print(f"Loaded {len(dataset)} samples")
print(f"Hallucinated: {labels.sum()} / {len(labels)}")

In [ ]:
# Cell 3: BERTScore Baseline (OPTIMIZED - load model once)
from bert_score import BERTScorer

print("Computing BERTScore baseline...")
print("Loading BERTScore model (once)...")
scorer = BERTScorer(lang="en", rescale_with_baseline=True, device="cuda")

bertscore_scores = []

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])[:2000]
    P, R, F1 = scorer.score([sample['answer']], [context_str])
    bertscore_scores.append(1 - F1.item())  # Lower similarity = higher hallucination score

bertscore_auroc = roc_auc_score(labels, bertscore_scores)
print(f"\nBERTScore AUROC: {bertscore_auroc:.4f}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("Loading NLI model (stable version)...")

model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
model.eval()

def compute_nli_score(context, answer):
    # STRONG truncation (important)
    context_str = " ".join(context)[:300]

    inputs = tokenizer(
        context_str,
        answer,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]

    # roberta-mnli label order: [contradiction, neutral, entailment]
    contradiction = probs[0].item()
    neutral = probs[1].item()

    # better scoring than your version
    return contradiction + 0.5 * neutral


print("Computing NLI baseline...")

nli_scores = []
valid_indices = []

for i, sample in enumerate(tqdm(dataset)):
    try:
        score = compute_nli_score(sample['context'], sample['answer'])
        nli_scores.append(score)
        valid_indices.append(i)
    except Exception:
        # DO NOT assign 0.5
        continue

nli_scores = np.array(nli_scores)
labels_matched = labels[valid_indices]

print(f"Valid samples: {len(nli_scores)} / {len(dataset)}")

nli_auroc = roc_auc_score(labels_matched, nli_scores)
print(f"\nNLI AUROC: {nli_auroc:.4f}")

In [ ]:
# Cell 5: Compute SDS Baseline
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Computing SDS baseline...")
sds_scores = []

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])
    answer = sample['answer']

    # Compute embeddings
    context_emb = embed_model.encode(context_str, convert_to_numpy=True)
    answer_emb = embed_model.encode(answer, convert_to_numpy=True)

    # SDS = L2 distance between context and answer embeddings
    sds = np.linalg.norm(context_emb - answer_emb)
    sds_scores.append(sds)

sds_auroc = roc_auc_score(labels, sds_scores)
print(f"\nSDS AUROC: {sds_auroc:.4f}")

In [ ]:
# Cell 6: SelfCheckGPT Baseline
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading model for SelfCheckGPT...")
model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda")

def generate_sample(prompt, temperature=1.0):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            temperature=temperature,
            do_sample=(temperature > 0.0),  # CRITICAL FIX: greedy when temp=0, stochastic when temp>0
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("Computing SelfCheckGPT baseline (this takes ~1-2 hours)...")
selfcheck_scores = []
n_samples = 3  # Number of sampled responses (increased from 3 for stability)

# Reuse the BERTScorer from Cell 3 (avoid reloading model ~11,000 times)
if 'scorer' not in locals():
    print("Creating BERTScorer for SelfCheckGPT...")
    from bert_score import BERTScorer
    scorer = BERTScorer(lang="en", rescale_with_baseline=True, device="cuda")

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])[:1500]
    prompt = f"Context: {context_str}\n\nQuestion: {sample['question']}\n\nAnswer:"

    # Generate deterministic original answer (temperature=0.0 → greedy decoding)
    original_answer = generate_sample(prompt, temperature=0.0)

    # Generate multiple stochastic samples (temperature=0.7 → sampling)
    sampled_answers = [generate_sample(prompt, temperature=0.7) for _ in range(n_samples)]

    # Compute consistency using REUSED scorer (BATCH processing - much faster)
    if any(s.strip() for s in sampled_answers):
        # Filter empty samples
        valid_samples = [s if s.strip() else "[EMPTY]" for s in sampled_answers]

        # Batch scoring (eliminates inner loop + model reloading)
        P, R, F1 = scorer.score(
            [original_answer] * len(valid_samples),
            valid_samples
        )
        inconsistency_scores = [1 - f.item() for f in F1]
    else:
        inconsistency_scores = [0.5] * n_samples

    selfcheck_scores.append(np.mean(inconsistency_scores))

selfcheck_auroc = roc_auc_score(labels, selfcheck_scores)
print(f"\nSelfCheckGPT AUROC: {selfcheck_auroc:.4f}")

In [ ]:
# Cell 7: Final Comparison

print("\n" + "="*50)
print(f"BASELINE COMPARISON ({DATASET.upper()})")
print("="*50)
print(f"\nMethod               AUROC")
print("-"*35)
print(f"NLI (RoBERTa-MNLI):  {nli_auroc:.4f}")
print(f"SDS (ours):          {sds_auroc:.4f}")
print(f"BERTScore:           {bertscore_auroc:.4f}")

print("-"*35)
print("\nNote: SDS + Uncertainty results in separate notebook (09-uncertainty-drift-kaggle.ipynb)")
print("="*50)